# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boddulavinaykumar6-stack/Flyrank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-05 — Feature Leakage Check

This notebook builds the feature vector for the selected FlyRank lane and tests whether any features contain information that would not have been available at the prediction moment.

The analysis uses March 2026 historical performance as the feature window and April 2026 performance as the future outcome window.

The target is whether a content item's average Google Search clicks increase from March to April 2026.

## 1. Build the feature vector

March 2026 is used as the historical feature window. April 2026 is kept separate as the future outcome window.

The feature vector contains five historical performance measures:

1. Average impressions
2. Average clicks
3. Average search position
4. Average pageviews
5. Average sessions

These features are calculated from March data only, so they are available before the April outcome is observed.

In [14]:
# W3 — Build the feature vector
# This cell also loads the warehouse so the notebook runs independently.

!pip -q install huggingface_hub duckdb pyarrow pandas scikit-learn

from google.colab import userdata
from huggingface_hub import login
import duckdb
import pandas as pd
import numpy as np

# Load Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Authenticate
login(token=HF_TOKEN)

# Create DuckDB connection
con = duckdb.connect()

# Give DuckDB access to the gated warehouse
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Hugging Face authentication successful.")
print("DuckDB connection ready.")


# ---------------------------------------------------------
# March 2026 = feature window
# ---------------------------------------------------------

march_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks,
    AVG(gsc_avg_position) AS avg_position,
    AVG(ga4_pageviews) AS avg_pageviews,
    AVG(ga4_sessions) AS avg_sessions
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id
""").df()


# GA4 values can be unavailable for some content/client combinations.
# Preserve the five-feature structure used in W5.
march_df[["avg_pageviews", "avg_sessions"]] = (
    march_df[["avg_pageviews", "avg_sessions"]].fillna(0)
)

feature_cols = [
    "avg_impressions",
    "avg_clicks",
    "avg_position",
    "avg_pageviews",
    "avg_sessions"
]

X_features = march_df[feature_cols]

print("\nFeature frame shape:", X_features.shape)
print("\nFeature columns:")
print(feature_cols)

print("\nFeature frame:")
display(X_features.head())

Hugging Face authentication successful.
DuckDB connection ready.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Feature frame shape: (331437, 5)

Feature columns:
['avg_impressions', 'avg_clicks', 'avg_position', 'avg_pageviews', 'avg_sessions']

Feature frame:


,avg_impressions,avg_clicks,avg_position,avg_pageviews,avg_sessions
0,210.419355,0.225806,7.209549,0.090909,0.090909
1,14.612903,0.000000,2.987198,0.000000,0.000000
2,181.612903,0.193548,6.724039,0.545455,0.272727
3,159.483871,0.419355,7.244844,0.181818,0.181818
4,1.354839,0.000000,14.432540,0.727273,0.636364


## 2. Feature notes

| Feature | Why it is available at the decision moment |
|---|---|
| `avg_impressions` | It is calculated from March Search Console observations, which are available before the April outcome. |
| `avg_clicks` | It is calculated from historical March clicks and therefore does not use April information. |
| `avg_position` | It represents historical March search position and is available before April. |
| `avg_pageviews` | It is calculated from March Analytics observations and is available before the April outcome. |
| `avg_sessions` | It is calculated from March Analytics observations and is available before the April outcome. |

`client_hash_id` and `content_hash_id` are retained for joining and grouping but are not predictive features.

In [15]:
# W3 — Feature notes
# Print the meaning, missing-value handling, and timing of every feature.

feature_notes = {
    "avg_impressions": {
        "meaning": "Average Google Search impressions during March 2026.",
        "missing": "No missing values after aggregation.",
        "available_when": "Available before the April prediction window because it uses March observations only."
    },
    "avg_clicks": {
        "meaning": "Average Google Search clicks during March 2026.",
        "missing": "No missing values after aggregation.",
        "available_when": "Available before the April prediction window because it uses March observations only."
    },
    "avg_position": {
        "meaning": "Average Google Search position during March 2026.",
        "missing": "No missing values after aggregation.",
        "available_when": "Available before the April prediction window because it uses March observations only."
    },
    "avg_pageviews": {
        "meaning": "Average GA4 pageviews during March 2026.",
        "missing": "Missing aggregated GA4 values are filled with 0 for this feature frame.",
        "available_when": "Available before April when GA4 data is available for the relevant records."
    },
    "avg_sessions": {
        "meaning": "Average GA4 sessions during March 2026.",
        "missing": "Missing aggregated GA4 values are filled with 0 for this feature frame.",
        "available_when": "Available before April when GA4 data is available for the relevant records."
    }
}

for feature, notes in feature_notes.items():
    print(f"\n{feature}")
    print(f"  Meaning: {notes['meaning']}")
    print(f"  Missing: {notes['missing']}")
    print(f"  Available when: {notes['available_when']}")

print("\nFeature count:", len(feature_notes))


avg_impressions
  Meaning: Average Google Search impressions during March 2026.
  Missing: No missing values after aggregation.
  Available when: Available before the April prediction window because it uses March observations only.

avg_clicks
  Meaning: Average Google Search clicks during March 2026.
  Missing: No missing values after aggregation.
  Available when: Available before the April prediction window because it uses March observations only.

avg_position
  Meaning: Average Google Search position during March 2026.
  Missing: No missing values after aggregation.
  Available when: Available before the April prediction window because it uses March observations only.

avg_pageviews
  Meaning: Average GA4 pageviews during March 2026.
  Missing: Missing aggregated GA4 values are filled with 0 for this feature frame.
  Available when: Available before April when GA4 data is available for the relevant records.

avg_sessions
  Meaning: Average GA4 sessions during March 2026.
  Missin

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [16]:
# W3 — Leakage hunt
# Compare an honest feature set with a deliberately leaked feature set.

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


# ---------------------------------------------------------
# Build the future outcome
# ---------------------------------------------------------

april_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_clicks) AS avg_clicks_apr
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-04/*.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id
""").df()


# Match March features with April outcome
model_df = march_df.merge(
    april_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Target used in our W5 modeling:
# 1 = April average clicks increased relative to March.
model_df["target"] = (
    model_df["avg_clicks_apr"] > model_df["avg_clicks"]
).astype(int)

print("Matched March-April rows:", len(model_df))

print("\nTarget distribution:")
display(model_df["target"].value_counts().to_frame("count"))


# ---------------------------------------------------------
# Honest model
# ---------------------------------------------------------

X_honest = model_df[feature_cols]
y = model_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_accuracy = accuracy_score(
    y_test,
    honest_pred
)


# ---------------------------------------------------------
# Deliberate leakage
# ---------------------------------------------------------
# avg_clicks_apr belongs to the future outcome window.
# It is also directly used to construct the target.

leaky_features = feature_cols + ["avg_clicks_apr"]

X_leaky = model_df[leaky_features]

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leaky_model.fit(
    X_train_leaky,
    y_train_leaky
)

leaky_pred = leaky_model.predict(
    X_test_leaky
)

leaky_accuracy = accuracy_score(
    y_test_leaky,
    leaky_pred
)


# ---------------------------------------------------------
# Compare
# ---------------------------------------------------------

leakage_results = pd.DataFrame({
    "model": [
        "Honest features",
        "Features + future leakage"
    ],
    "accuracy": [
        honest_accuracy,
        leaky_accuracy
    ]
})

display(leakage_results)

print(
    f"\nAccuracy increase from leakage: "
    f"{leaky_accuracy - honest_accuracy:.4f}"
)

print("\nLeaked feature:")
print("avg_clicks_apr")

print(
    "\nReason it is leakage: "
    "April clicks are part of the future outcome window and "
    "are directly used to define the target."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Matched March-April rows: 331436

Target distribution:


,count
target,
0,293965
1,37471


,model,accuracy
0,Honest features,0.886948
1,Features + future leakage,0.952178



Accuracy increase from leakage: 0.0652

Leaked feature:
avg_clicks_apr

Reason it is leakage: April clicks are part of the future outcome window and are directly used to define the target.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [17]:
# W3 — Fields excluded from the predictive feature set

excluded_fields = {
    "avg_clicks_apr": (
        "Future April information; it belongs to the outcome window "
        "and directly contributes to the target."
    ),
    "target": (
        "The label itself; using it as a feature would directly reveal "
        "the answer."
    ),
    "client_hash_id": (
        "Pseudonymous identifier; retained for joining/grouping only, "
        "not used as a predictive feature."
    ),
    "content_hash_id": (
        "Pseudonymous identifier; retained for joining only, "
        "not used as a predictive feature."
    )
}

print("Excluded fields and reasons:\n")

for field, reason in excluded_fields.items():
    print(f"- {field}: {reason}")


# Final predictive feature list
final_features = [
    "avg_impressions",
    "avg_clicks",
    "avg_position",
    "avg_pageviews",
    "avg_sessions"
]

print("\nFinal predictive features:")
for feature in final_features:
    print(f"- {feature}")

print("\nFinal feature count:", len(final_features))

Excluded fields and reasons:

- avg_clicks_apr: Future April information; it belongs to the outcome window and directly contributes to the target.
- target: The label itself; using it as a feature would directly reveal the answer.
- client_hash_id: Pseudonymous identifier; retained for joining/grouping only, not used as a predictive feature.
- content_hash_id: Pseudonymous identifier; retained for joining only, not used as a predictive feature.

Final predictive features:
- avg_impressions
- avg_clicks
- avg_position
- avg_pageviews
- avg_sessions

Final feature count: 5


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.